# Etapa 2 — MLP inicial

Classificação de BOs envolvendo celulares subtraídos em São Paulo capital. 0=FURTO, 1=ROUBO. O modelo classifica características de uma ocorrência; não prevê crimes futuros. Esta única arquitetura usa treino em 2023 e validação em 2024. O teste de 2025 permanece sem avaliação. O notebook consome os resultados do treinamento real, sem duplicar o pipeline pesado.

In [1]:
from pathlib import Path
import sys,json,csv
ROOT=Path.cwd()
if not (ROOT/'models').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from ml.runtime import OUT,FEATURES
config=json.loads((ROOT/'models/config_modelo.json').read_text(encoding='utf-8'))
print('Atributos:', FEATURES)
print('Treino:',config['train_rows'],'de 2023; validação:',config['validation_rows'],'de 2024')
print('2025 isolado:',config['test_rows_isolated'])


Atributos: ['mes', 'dia_semana', 'tipo_local', 'bairro']
Treino: 190124 de 2023; validação: 158895 de 2024
2025 isolado: 156168


## Integridade e escolha de atributos

O índice e a chave dos metadados são confrontados com os valores da base analítica, antes de dividir por data do fato. Não há sobreposição de boletins. Tuplas de atributos iguais podem pertencer a BOs distintos; não foram eliminadas. `periodo_fato`, hora e indicadores de preenchimento são excluídos por risco de proxy administrativo. Nenhum campo jurídico ou identificador entra na rede.

In [2]:
validation=json.loads((OUT/'validacao_dados.json').read_text(encoding='utf-8'))
print(validation['checks'])
print(validation['duplicidade_contextos'])


{'linhas_base': True, 'linhas_analiticas': True, 'indice_unico_contiguo': True, 'chaves_unicas': True, 'chaves_analiticas_unicas': True, 'sem_chaves_ausentes': True, 'alinhamento_celula_a_celula': True, 'rotulos_binarios': True, 'intervalos': True, 'grupos_sem_sobreposicao': True}
[{'ano': 2023, 'tuplas_X_distintas': 45017, 'repeticoes_X_entre_BOs': 145107, 'tuplas_X_com_ambas_classes': 12274}, {'ano': 2024, 'tuplas_X_distintas': 42372, 'repeticoes_X_entre_BOs': 116523, 'tuplas_X_com_ambas_classes': 11452}]


## Pré-processamento

One-Hot nominal para as quatro entradas. Vocabulários e frequências vêm apenas de 2023. Bairros com menos de 50 observações formam um bucket estatístico de raros, também usado para bairros novos. Tipos de local novos usam a representação de ausente. Só equivalências explícitas de acentuação verificadas no treino são unificadas. Ausências não removem linhas. CSR float32 é densificado apenas por lote de 256. Não há balanceamento nem escalonamento.

In [3]:
prep=json.loads((OUT/'preprocessamento.json').read_text(encoding='utf-8'))
for name,stats in prep.items():print(name,stats)


treino {'rows': 190124, 'dimensao': 299, 'bairros_observados_treino': 1913, 'bairros_retidos': 247, 'aliases_ativados': {'REPÚBLICA': 'REPUBLICA', 'CONSOLAÇÃO': 'CONSOLACAO', 'BRÁS': 'BRAS', 'SÉ': 'SE', 'SACOMÃ': 'SACOMA', 'FREGUESIA DO Ó': 'FREGUESIA DO O', 'VILA JACUÍ': 'VILA JACUI', 'VILA OLÍMPIA': 'VILA OLIMPIA', 'SÃO MATEUS': 'SAO MATEUS', 'VILA SÔNIA': 'VILA SONIA', 'CAPÃO REDONDO': 'CAPAO REDONDO', 'GRAJAÚ': 'GRAJAU', 'ÁGUA RASA': 'AGUA RASA', 'CIDADE LÍDER': 'CIDADE LIDER', 'JAGUARÁ': 'JAGUARA', 'SÃO RAFAEL': 'SAO RAFAEL'}, 'limiar_bairro': 50, 'mes': {'ausentes_entrada': 0, 'desconhecidos': 0, 'raros': 0}, 'dia_semana': {'ausentes_entrada': 0, 'desconhecidos': 0, 'raros': 0}, 'tipo_local': {'ausentes_entrada': 14031, 'desconhecidos': 0, 'raros': 0}, 'bairro': {'ausentes_entrada': 633, 'desconhecidos': 0, 'raros': 14244}}
validacao {'rows': 158895, 'dimensao': 299, 'bairros_observados_treino': 1913, 'bairros_retidos': 247, 'aliases_ativados': {'REPÚBLICA': 'REPUBLICA', 'CONSOLA

## Arquitetura e execução

Uma camada Dense de 64 neurônios ReLU e saída sigmoid. Binary Cross-Entropy, Adam 0,001, batch 256, máximo 30 épocas, Early Stopping val_loss com paciência 5 e melhores pesos restaurados; semente 42. O comando de reprodução é `python -m ml.train`, com PYTHONPATH=src e dependências de requirements-ml.txt. Não há busca de hiperparâmetros neste notebook.

In [4]:
for name in ['input_dimension','trainable_parameters','epochs_ran','best_epoch','stop_reason','training_seconds','tensorflow_devices','dependencies']:
    print(name,':',config[name])


input_dimension : 299
trainable_parameters : 19265
epochs_ran : 10
best_epoch : 5
stop_reason : early_stopping_val_loss
training_seconds : 28.452553099981742
tensorflow_devices : [{'name': '/physical_device:CPU:0', 'type': 'CPU'}]
dependencies : {'tensorflow': '2.20.0', 'keras': '3.11.3', 'scikit-learn': '1.7.2', 'joblib': '1.5.2', 'numpy': '2.2.6', 'scipy': '1.18.1', 'pyarrow': '26.0.1', 'duckdb': '2.0.0', 'matplotlib': '3.11.2'}


![Curvas reais de treino](../reports/figures/curvas_mlp_baseline.png)

A validação escolhe a melhor época. A diferença entre treino e validação pode reunir sobreajuste e mudança temporal; ela não prova isoladamente uma das causas. Contextos idênticos com duas classes também limitam o desempenho possível com as entradas disponíveis.

## Métricas e referência

A referência sempre prevê a classe majoritária observada em 2023. Precision, recall e F1 do resumo têm ROUBO como classe positiva. Macro F1 e balanced accuracy ponderam igualmente as classes. As métricas por classe e a matriz evidenciam erros que uma acurácia isolada pode esconder.

In [5]:
with (ROOT/'reports/tables/metricas_baseline.csv').open(encoding='utf-8-sig') as f:
    for row in csv.DictReader(f):print(row)
with (ROOT/'reports/tables/metricas_baseline_por_classe.csv').open(encoding='utf-8-sig') as f:
    for row in csv.DictReader(f):
        if row['ano']=='2024':print(row)


{'modelo': 'MLP', 'ano': '2023', 'n': '190124', 'accuracy': '0.7014527361090657', 'precision': '0.6778349834983498', 'recall': '0.6134308208785826', 'f1': '0.6440267665079992', 'macro_f1': '0.6934755993367914', 'balanced_accuracy': '0.692057589144622'}
{'modelo': 'MLP', 'ano': '2024', 'n': '158895', 'accuracy': '0.6872840555083546', 'precision': '0.6327374042256791', 'recall': '0.6115798922800718', 'f1': '0.6219787743923314', 'macro_f1': '0.6776641034482787', 'balanced_accuracy': '0.6769159034481669'}
{'modelo': 'Majoritario', 'ano': '2024', 'n': '158895', 'accuracy': '0.5793448503728877', 'precision': '0.0', 'recall': '0.0', 'f1': '0.0', 'macro_f1': '0.3668260609683204', 'balanced_accuracy': '0.5'}
{'modelo': 'MLP', 'ano': '2024', 'classe': 'FURTO', 'precision': '0.7246579700922685', 'recall': '0.742251914616262', 'f1': '0.733349432504226', 'suporte': '92055', 'previsoes': '94290'}
{'modelo': 'MLP', 'ano': '2024', 'classe': 'ROUBO', 'precision': '0.6327374042256791', 'recall': '0.6115

![Matriz de confusão — 2024](../reports/figures/matriz_confusao_baseline.png)

## Recarga e inferência Python

O exemplo abaixo é hipotético e usa o artefato salvo. Retorna as duas probabilidades estimadas, sem afirmar calibração. Modelos e joblib devem ser locais e confiáveis. Nenhuma chamada usa dados de 2025.

In [6]:
from ml.inference import ClassificadorOcorrencias
classificador=ClassificadorOcorrencias()
print(classificador.classificar(mes=5,dia_semana=2,tipo_local='VIA PÚBLICA',bairro='BELA VISTA'))
print((OUT/'consistencia_recarga.json').read_text(encoding='utf-8'))
print((OUT/'testes_etapa2.json').read_text(encoding='utf-8'))


{'codigo': 0, 'classe': 'FURTO', 'probabilidade_furto': 0.7717520743608475, 'probabilidade_roubo': 0.22824792563915253, 'probabilidades_calibradas': False}
{
  "validation_rows": 158895,
  "max_abs_difference": 0.0,
  "passed": true
}
[
  {
    "arquivo": "test_integridade.py",
    "executados": 8,
    "falhas": 0,
    "erros": 0,
    "sucesso": true
  },
  {
    "arquivo": "test_ml_pipeline.py",
    "executados": 15,
    "falhas": 0,
    "erros": 0,
    "sucesso": true
  }
]


## Limitações e próxima etapa

O domínio é de celulares subtraídos e BOs elegíveis com natureza não ambígua; não há generalização demonstrada para todos os crimes. Local/bairro podem refletir práticas de registro, apesar de não revelarem diretamente a natureza. Investigar grupos de erro, raridade, estabilidade temporal e calibração em desenvolvimento, mantendo 2025 fechado. Não se inicia comparação de arquiteturas nem aplicação web. Ver o relatório completo em `../reports/relatorio_etapa2.md` e a reprodução em `../docs/etapa2_reproducao.md`.